# Notebook 09 — VCF Format: Reading, Filtering, and Annotation

**Module:** 09 — Genomics and NGS  
**Notebook:** 09 of 16  
**Time estimate:** 75 minutes

> The VCF format is the universal output of variant callers. You must be able to
> parse, filter, and annotate VCF files from scratch — and explain every field.

---
## Step 1 — Motivation

After HaplotypeCaller produces a VCF, that VCF needs to be filtered, annotated,
and interpreted. Raw variant calls include artefacts; filtering separates real
variants from noise. Annotation adds biological meaning (gene name, protein change,
population frequency) to each variant.

---
## Step 2 — Intuition

**VCF structure:**
- `##` header lines: metadata (file format, reference, contig lengths, INFO/FORMAT field definitions)
- `#CHROM` line: column headers including sample names
- Data lines: one variant per line, 8 mandatory columns + FORMAT + per-sample columns

**The 8 mandatory columns:**
1. `CHROM` — chromosome
2. `POS` — 1-based position of first base in REF
3. `ID` — variant identifier (rsID or `.`)
4. `REF` — reference allele(s)
5. `ALT` — alternate allele(s), comma-separated if multi-allelic
6. `QUAL` — Phred-scaled call quality
7. `FILTER` — PASS or filter names that failed
8. `INFO` — semicolon-separated key=value annotations

**FORMAT and sample columns:**
- `FORMAT` defines the fields for each sample (e.g., `GT:AD:DP:GQ:PL`)
- Each sample column has corresponding values in the same order

---
## Step 3 — Biological Background

**Variant annotation layers (in order of biological relevance):**

1. **Genomic context:** exonic, intronic, UTR, intergenic, splice site
2. **Functional consequence:** synonymous, missense, nonsense, frameshift, splice-disrupting
3. **Conservation:** GERP score, PhyloP — is this position conserved across species?
4. **Population frequency:** gnomAD allele frequency — common (>1%) or rare (<0.1%)?
5. **Pathogenicity prediction:** SIFT, PolyPhen-2, CADD score — computational effect prediction
6. **Clinical significance:** ClinVar classification (benign, likely benign, VUS, likely pathogenic, pathogenic)

**GATK variant filtration approaches:**
- **Hard filtering:** Apply fixed thresholds (QD < 2, FS > 60, MQ < 40)
- **VQSR (Variant Quality Score Recalibration):** Train a Gaussian mixture model on
  known variant databases; assign posterior probability of being a true variant.
  Requires large cohorts (>30 samples or >1000 variants)

---
## Step 4 — Mathematical Explanation

**Key INFO fields from GATK HaplotypeCaller:**

- **QD (Quality by Depth):** $QD = QUAL / DP$. Normalizes variant quality by depth;
  low QD suggests the quality is only high because of depth, not because the variant
  is convincing. Filter: $QD < 2.0$

- **FS (Fisher Strand bias):** Phred-scaled p-value from Fisher's exact test on the
  strand distribution of reads supporting REF vs ALT:
  $$FS = -10 \log_{10}(p_{\text{Fisher}})$$
  High FS = significant strand bias = likely artefact. Filter: $FS > 60$ (SNPs),
  $FS > 200$ (indels)

- **MQ (Mapping Quality):** RMS mapping quality of reads at the site. Low MQ = reads
  are misaligned. Filter: $MQ < 40$

- **SOR (Symmetric Odds Ratio):** Alternative to FS that is less sensitive to
  coverage. Filter: $SOR > 3.0$

In [ ]:
# Step 6 — Python: VCF parser from scratch
from dataclasses import dataclass, field
from typing import Iterator
import re


@dataclass
class VcfRecord:
    chrom: str
    pos: int
    id: str
    ref: str
    alt: list[str]
    qual: float | None
    filter: list[str]
    info: dict
    format_keys: list[str]
    samples: list[dict]

    @property
    def is_snp(self) -> bool:
        return all(len(a) == 1 and len(self.ref) == 1 for a in self.alt)

    @property
    def is_indel(self) -> bool:
        return any(len(a) != len(self.ref) for a in self.alt)

    @property
    def is_multiallelic(self) -> bool:
        return len(self.alt) > 1

    @property
    def passed(self) -> bool:
        return self.filter == ['PASS'] or self.filter == []

    def vaf(self, sample_idx: int = 0) -> float | None:
        s = self.samples[sample_idx] if sample_idx < len(self.samples) else {}
        ad = s.get('AD')
        if ad is None:
            return None
        parts = [int(x) for x in str(ad).split(',')]
        if len(parts) < 2 or sum(parts) == 0:
            return None
        return parts[1] / sum(parts)


def parse_info(info_str: str) -> dict:
    """Parse VCF INFO field string into dict."""
    if info_str == '.':
        return {}
    result = {}
    for item in info_str.split(';'):
        if '=' in item:
            k, v = item.split('=', 1)
            # Try to convert to float/int
            try:
                result[k] = int(v) if '.' not in v else float(v)
            except ValueError:
                result[k] = v
        else:
            result[item] = True  # flag field
    return result


def parse_sample(format_str: str, sample_str: str) -> dict:
    keys = format_str.split(':')
    vals = sample_str.split(':')
    result = {}
    for k, v in zip(keys, vals):
        if v == '.' or v == './.': result[k] = None
        else: result[k] = v
    return result


def parse_vcf(vcf_text: str) -> tuple[list[str], list[VcfRecord]]:
    """Parse a VCF string. Returns (header_lines, records)."""
    headers = []
    records = []
    sample_names = []

    for line in vcf_text.strip().split('\n'):
        if line.startswith('##'):
            headers.append(line)
        elif line.startswith('#CHROM'):
            cols = line.split('\t')
            sample_names = cols[9:] if len(cols) > 9 else []
            headers.append(line)
        else:
            cols = line.split('\t')
            if len(cols) < 8:
                continue
            qual = float(cols[5]) if cols[5] != '.' else None
            filter_field = cols[6].split(';') if cols[6] != '.' else []
            fmt_keys = cols[8].split(':') if len(cols) > 8 else []
            samples = [parse_sample(cols[8], cols[9+i]) for i in range(len(sample_names))] if len(cols) > 9 else []

            records.append(VcfRecord(
                chrom=cols[0],
                pos=int(cols[1]),
                id=cols[2],
                ref=cols[3],
                alt=cols[4].split(','),
                qual=qual,
                filter=filter_field,
                info=parse_info(cols[7]),
                format_keys=fmt_keys,
                samples=samples,
            ))
    return headers, records


# Test VCF content
test_vcf = """##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##INFO=<ID=QD,Number=1,Type=Float,Description="Quality by Depth">
##INFO=<ID=FS,Number=1,Type=Float,Description="Fisher Strand">
##INFO=<ID=MQ,Number=1,Type=Float,Description="Mapping Quality">
##INFO=<ID=DP,Number=1,Type=Integer,Description="Depth">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allele Depth">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Depth">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=PL,Number=G,Type=Integer,Description="Phred Likelihoods">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	NA12878
chr1	100	rs1234	A	T	50.2	PASS	QD=5.2;FS=1.2;MQ=58.5;DP=30	GT:AD:DP:GQ:PL	0/1:15,15:30:50:50,0,50
chr1	200	.	C	G	12.1	low_qual	QD=1.5;FS=45.2;MQ=38.0;DP=8	GT:AD:DP:GQ:PL	0/1:5,3:8:12:12,0,30
chr1	300	.	ATG	A	80.0	PASS	QD=8.0;FS=2.1;MQ=60.0;DP=40	GT:AD:DP:GQ:PL	0/1:20,20:40:80:80,0,80
chr1	400	.	G	A,T	35.0	PASS	QD=3.5;FS=8.0;MQ=55.0;DP=20	GT:AD:DP:GQ:PL	0/1:10,7,3:20:35:35,0,20,50,10,60
"""

headers, records = parse_vcf(test_vcf)
print(f"Parsed {len(records)} variants\n")
for r in records:
    vaf = r.vaf()
    print(f"  {r.chrom}:{r.pos} {r.ref}>{','.join(r.alt)} "
          f"QUAL={r.qual} FILTER={r.filter} "
          f"SNP={r.is_snp} INDEL={r.is_indel} "
          f"VAF={vaf:.2f}" if vaf else f"VAF=None")
    print(f"    INFO: {r.info}")

In [ ]:
# GATK hard filtering
def gatk_hard_filter(
    records: list[VcfRecord],
    snp_filters: dict = None,
    indel_filters: dict = None,
) -> list[VcfRecord]:
    """
    Apply GATK recommended hard filters.
    Returns records with FILTER field updated.
    """
    if snp_filters is None:
        snp_filters = {
            'QD': ('<', 2.0),
            'FS': ('>', 60.0),
            'MQ': ('<', 40.0),
            'SOR': ('>', 3.0),
        }
    if indel_filters is None:
        indel_filters = {
            'QD': ('<', 2.0),
            'FS': ('>', 200.0),
            'SOR': ('>', 10.0),
        }

    filtered = []
    for r in records:
        filters = list(r.filter) if r.filter != ['PASS'] else []
        active_filters = snp_filters if r.is_snp else indel_filters
        for field_name, (op, threshold) in active_filters.items():
            val = r.info.get(field_name)
            if val is None:
                continue
            if op == '<' and float(val) < threshold:
                filters.append(f'{field_name}_lt_{threshold}')
            elif op == '>' and float(val) > threshold:
                filters.append(f'{field_name}_gt_{threshold}')
        r.filter = filters if filters else ['PASS']
        filtered.append(r)
    return filtered


filtered_records = gatk_hard_filter(records)
print("After GATK hard filtering:")
for r in filtered_records:
    status = 'PASS' if r.passed else 'FAIL'
    print(f"  {r.chrom}:{r.pos} {r.ref}>{','.join(r.alt)} → {status} ({r.filter})")

passed = [r for r in filtered_records if r.passed]
print(f"\n{len(passed)}/{len(filtered_records)} variants passed filtering")

In [ ]:
# Step 7 — Visualization: VCF statistics
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import fisher_exact

# Simulate a larger variant set for visualization
rng = np.random.default_rng(42)
n_variants = 500

# Simulate realistic INFO field distributions
true_snp_QD = rng.normal(12, 4, 400).clip(0, 40)
false_snp_QD = rng.exponential(1.5, 100).clip(0, 40)
all_QD = np.concatenate([true_snp_QD, false_snp_QD])

true_snp_FS = rng.exponential(3, 400)
false_snp_FS = rng.exponential(25, 100)
all_FS = np.concatenate([true_snp_FS, false_snp_FS])

all_MQ = np.concatenate([rng.normal(58, 3, 400).clip(0, 60), rng.normal(30, 8, 100).clip(0, 60)])
is_true = np.array([True]*400 + [False]*100)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: QD distribution
ax = axes[0, 0]
ax.hist(all_QD[is_true], bins=30, alpha=0.6, color='green', label='True variants')
ax.hist(all_QD[~is_true], bins=30, alpha=0.6, color='red', label='Artefacts')
ax.axvline(2.0, color='black', ls='--', lw=2, label='GATK filter QD<2')
ax.set_xlabel('QD (Quality by Depth)')
ax.set_ylabel('Count')
ax.set_title('A. QD distribution: true vs. artefact variants')
ax.legend(fontsize=8)

# Panel B: FS distribution
ax2 = axes[0, 1]
ax2.hist(all_FS[is_true], bins=40, range=(0,200), alpha=0.6, color='green', label='True variants')
ax2.hist(all_FS[~is_true], bins=40, range=(0,200), alpha=0.6, color='red', label='Artefacts')
ax2.axvline(60.0, color='black', ls='--', lw=2, label='GATK filter FS>60')
ax2.set_xlabel('FS (Fisher Strand)')
ax2.set_ylabel('Count')
ax2.set_title('B. Strand bias (FS): true vs. artefact variants')
ax2.legend(fontsize=8)

# Panel C: MQ distribution
ax3 = axes[1, 0]
ax3.hist(all_MQ[is_true], bins=30, alpha=0.6, color='green', label='True variants')
ax3.hist(all_MQ[~is_true], bins=30, alpha=0.6, color='red', label='Artefacts')
ax3.axvline(40.0, color='black', ls='--', lw=2, label='GATK filter MQ<40')
ax3.set_xlabel('MQ (Mapping Quality)')
ax3.set_ylabel('Count')
ax3.set_title('C. Mapping quality (MQ): true vs. artefact variants')
ax3.legend(fontsize=8)

# Panel D: Filter result summary
ax4 = axes[1, 1]
n_pass_true = ((all_QD[is_true] >= 2) & (all_FS[is_true] <= 60) & (all_MQ[is_true] >= 40)).sum()
n_fail_true = is_true.sum() - n_pass_true
n_pass_false = ((all_QD[~is_true] >= 2) & (all_FS[~is_true] <= 60) & (all_MQ[~is_true] >= 40)).sum()
n_fail_false = (~is_true).sum() - n_pass_false

categories = ['True variants\nPASS', 'True variants\nFAIL', 'Artefacts\nPASS', 'Artefacts\nFAIL']
values = [n_pass_true, n_fail_true, n_pass_false, n_fail_false]
colors4 = ['darkgreen', 'lightgreen', 'tomato', 'darkred']
bars4 = ax4.bar(categories, values, color=colors4, alpha=0.85)
for bar, val in zip(bars4, values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val), ha='center', fontsize=9)
ax4.set_ylabel('Number of variants')
ax4.set_title('D. Hard filter performance\n(GATK: QD≥2, FS≤60, MQ≥40)')

plt.suptitle('VCF Quality Filtering Statistics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('vcf_filtering.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Filter sensitivity: {n_pass_true}/{is_true.sum()} true variants pass ({100*n_pass_true/is_true.sum():.1f}%)')
print(f'Filter specificity: {n_fail_false}/{(~is_true).sum()} artefacts filtered ({100*n_fail_false/(~is_true).sum():.1f}%)')

---
## Step 8 — Exercises

1. Parse the VCF record `chr2\t500\t.\tA\tATG\t30.0\tPASS\tQD=3.0;FS=5.0;DP=20\tGT:AD:DP\t0/1:10,10:20`. Extract: is it a SNP or indel? What is the VAF? What does the genotype mean?
2. Implement `vcf_stats(records)` that returns total variants, SNP count, indel count, multi-allelic count, pass rate.
3. Write a VCF writer function `write_vcf(headers, records) -> str` that produces valid VCF text.

---
## Step 10 — Quiz

1. What are the 8 mandatory VCF columns?
2. What does `FILTER=PASS` mean vs. `FILTER=low_qual`?
3. How is QD computed and why is low QD suspicious?
4. What does the GT field `0/1` mean? What about `1/1`? `0/2`?
5. What is VQSR and when would you use it instead of hard filtering?

---
## Step 12 — Reflection

> *[In one paragraph: explain the difference between variant calling quality (QUAL)
> and genotype quality (GQ). Can a variant have high QUAL but low GQ?]*

---
*Next: `10_rnaseq_spliced_alignment_star.ipynb`*